# Chapter 19 — Prior-Data Fitted Networks

Reproduces:
- Figure 19.1: PFN training curve on Bayesian-linear-regression episodes.
- Figure 19.2: PFN posterior mean vs Bayes-optimal posterior mean.
- Figure 19.3: Held-out MSE — PFN, Bayes-optimal, ridge, NW kernel.

The PFN here is a small two-layer transformer with self-attention over a
context+query token sequence. A single-layer kernel smoother cannot recover
the linear-regression posterior (which is *linear* in the context labels,
not a smoother), so we need depth — this matches the constructions studied
by Garg et al. 2022 and von Oswald et al. 2023 for ICL on linear regression.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from tabkernels.core.base import Prior
from tabkernels.training import PFNTrainer

torch.manual_seed(0); np.random.seed(0)
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
FIGURES_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)


## Tractable prior with closed-form Bayes posterior

We use a Bayesian linear-regression prior so we can compute the *true* posterior
predictive in closed form and compare the trained PFN against it.

Each episode:

1. Draw task weight $w \sim \mathcal{N}(0, \sigma_w^2 I_d)$.
2. Draw covariates $X \sim \mathcal{N}(0, I_d)$.
3. Draw labels $y = X w + \varepsilon$, $\varepsilon \sim \mathcal{N}(0, \sigma_\varepsilon^2)$.

The posterior over $w$ given $(X_\text{ctx}, y_\text{ctx})$ is Gaussian with closed form,
so the posterior predictive mean at $x_q$ is a closed-form linear function of
$y_\text{ctx}$.


In [ ]:
class LinearRegPrior(Prior):
    def __init__(self, sigma_w=1.0, sigma_eps=0.3):
        self.sigma_w = sigma_w
        self.sigma_eps = sigma_eps

    def sample_episode(self, n_ctx, n_query, d, seed=None):
        g = torch.Generator()
        if seed is not None:
            g.manual_seed(seed)
        w = self.sigma_w * torch.randn(d, generator=g)
        N = n_ctx + n_query
        X = torch.randn(N, d, generator=g)
        eps = self.sigma_eps * torch.randn(N, generator=g)
        y = X @ w + eps
        return X[:n_ctx], y[:n_ctx], X[n_ctx:], y[n_ctx:]


def bayes_posterior_mean(X_ctx, y_ctx, X_q, sigma_w=1.0, sigma_eps=0.3):
    """Closed-form Bayes posterior predictive mean for linear regression.

    posterior on w: N(mu, Sigma) with
        Sigma = (X^T X / sigma_eps^2 + I / sigma_w^2)^{-1}
        mu    = Sigma X^T y / sigma_eps^2
    predictive mean at x_q is x_q^T mu.
    """
    d = X_ctx.shape[1]
    XtX = X_ctx.T @ X_ctx
    prec = XtX / (sigma_eps ** 2) + torch.eye(d) / (sigma_w ** 2)
    cov = torch.linalg.inv(prec)
    mu = cov @ X_ctx.T @ y_ctx / (sigma_eps ** 2)
    return X_q @ mu


## A small transformer PFN

Two layers of self-attention over a sequence of `(context, query)` tokens.
Context tokens embed both $x_i$ and $y_i$; query tokens embed only $x_q$
(zero-padded $y$ slot). After two attention+MLP blocks, a linear head
reads off the prediction at each query token.

This is the minimal architecture under which ICL on linear regression
converges to the Bayes posterior in the limit of large context and many
training episodes (Garg et al. 2022, von Oswald et al. 2023).


In [ ]:
class TinyPFN(nn.Module):
    def __init__(self, d_in, d_emb=64, n_heads=4, n_layers=2, d_ff=128):
        super().__init__()
        self.x_embed = nn.Linear(d_in, d_emb)
        self.y_embed = nn.Linear(1, d_emb)
        self.q_marker = nn.Parameter(torch.randn(d_emb) * 0.02)  # query-token marker
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'ln1': nn.LayerNorm(d_emb),
                'attn': nn.MultiheadAttention(d_emb, n_heads, batch_first=True),
                'ln2': nn.LayerNorm(d_emb),
                'mlp': nn.Sequential(
                    nn.Linear(d_emb, d_ff), nn.GELU(), nn.Linear(d_ff, d_emb),
                ),
            }) for _ in range(n_layers)
        ])
        self.out_ln = nn.LayerNorm(d_emb)
        self.head = nn.Linear(d_emb, 1)

    def forward(self, X_q, X_ctx, y_ctx):
        ctx_tok = self.x_embed(X_ctx) + self.y_embed(y_ctx.unsqueeze(-1))
        q_tok = self.x_embed(X_q) + self.q_marker
        seq = torch.cat([ctx_tok, q_tok], dim=0).unsqueeze(0)  # (1, N_total, d_emb)
        n_c = X_ctx.shape[0]
        for layer in self.layers:
            normed = layer['ln1'](seq)
            attn_out, _ = layer['attn'](normed, normed, normed, need_weights=False)
            seq = seq + attn_out
            seq = seq + layer['mlp'](layer['ln2'](seq))
        return self.head(self.out_ln(seq[0, n_c:]))  # (n_q, 1)


## Train

3000 steps, $n_\text{ctx} = 64$, $n_\text{query} = 32$, $d = 4$.
Adam with linear warmup + cosine decay.


In [ ]:
D = 4
prior = LinearRegPrior(sigma_w=1.0, sigma_eps=0.3)
torch.manual_seed(0)
model = TinyPFN(d_in=D, d_emb=64, n_heads=4, n_layers=2, d_ff=128)

trainer = PFNTrainer(prior=prior, model=model, n_steps=3000,
                     n_ctx=64, n_query=32, d=D, lr=3e-3,
                     eval_every=100, seed=1)
trainer.optim = torch.optim.Adam(model.parameters(), lr=3e-3)
warmup_steps = 200
sched = torch.optim.lr_scheduler.LambdaLR(
    trainer.optim,
    lr_lambda=lambda s: min((s + 1) / warmup_steps, 0.5 * (1 + np.cos(np.pi * (s - warmup_steps) / max(1, trainer.n_steps - warmup_steps)))),
)

losses, eval_losses = [], []
for step in range(trainer.n_steps):
    losses.append(trainer.step(step))
    sched.step()
    if (step + 1) % 100 == 0:
        eval_losses.append(trainer.evaluate(n_episodes=16))

print(f'final train loss : {losses[-1]:.4f}')
print(f'final eval  loss : {eval_losses[-1]:.4f}')


## Figure 19.1: Training curve

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 3.5))
ax.plot(losses, alpha=0.25, color='C0', label='train (per step)')
window = 30
smooth = np.convolve(losses, np.ones(window) / window, mode='valid')
ax.plot(np.arange(window - 1, len(losses)), smooth, color='C0',
        label=f'train ({window}-step MA)')
eval_steps = np.arange(100, len(losses) + 1, 100)
ax.plot(eval_steps, eval_losses, 'o-', color='C1', label='eval (16 episodes)')
ax.set_xlabel('step'); ax.set_ylabel('MSE')
ax.set_title('Figure 19.1: PFN training curve on Bayesian-linear-regression episodes')
ax.legend(); ax.grid(alpha=0.3); ax.set_yscale('log')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_19_01_pfn_training_curve.pdf', bbox_inches='tight')
plt.show()


## Figure 19.2: PFN posterior mean tracks the Bayes-optimal posterior mean

For each held-out task we compute (i) the trained PFN's posterior predictive mean
and (ii) the closed-form Bayes-optimal posterior mean. If the PFN has internalised
the prior, the two should track on the diagonal.


In [ ]:
model.eval()
n_eval_tasks = 50
n_q_per_task = 16
pfn_preds = []
bayes_preds = []
y_truth = []
with torch.no_grad():
    for k in range(n_eval_tasks):
        X_c, y_c, X_q, y_q = prior.sample_episode(n_ctx=64, n_query=n_q_per_task,
                                                  d=D, seed=10000 + k)
        yhat = model(X_q, X_c, y_c).squeeze(-1)
        ybayes = bayes_posterior_mean(X_c, y_c, X_q,
                                      sigma_w=prior.sigma_w, sigma_eps=prior.sigma_eps)
        pfn_preds.append(yhat); bayes_preds.append(ybayes); y_truth.append(y_q)
pfn_preds = torch.cat(pfn_preds).numpy()
bayes_preds = torch.cat(bayes_preds).numpy()
y_truth = torch.cat(y_truth).numpy()

corr = np.corrcoef(pfn_preds, bayes_preds)[0, 1]

fig, ax = plt.subplots(1, 1, figsize=(5.5, 5.5))
lim = max(abs(pfn_preds).max(), abs(bayes_preds).max()) * 1.05
ax.plot([-lim, lim], [-lim, lim], '--', color='gray', alpha=0.6, label='identity')
ax.scatter(bayes_preds, pfn_preds, s=10, alpha=0.5)
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel('Bayes-optimal posterior mean')
ax.set_ylabel('PFN posterior mean')
ax.set_title(f'Figure 19.2: PFN tracks Bayes optimum (corr = {corr:.3f})')
ax.legend(loc='upper left'); ax.grid(alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_19_02_pfn_vs_bayes.pdf', bbox_inches='tight')
plt.show()
print(f'PFN <-> Bayes correlation: {corr:.4f}')


## Figure 19.3: Held-out MSE — PFN vs Bayes vs ridge vs NW

Four predictors on the same held-out tasks:

- PFN: trained transformer, no per-task fitting at test time.
- Bayes-optimal: closed-form posterior predictive mean (the irreducible target).
- Ridge: per-task ridge regression with the matched prior precision $\lambda = \sigma_\varepsilon^2 / \sigma_w^2$. *Requires per-task fitting*.
- NW: Nadaraya-Watson kernel smoother (no prior knowledge, no per-task fitting).


In [ ]:
def ridge_predict(X_c, y_c, X_q, lam):
    d = X_c.shape[1]
    A = X_c.T @ X_c + lam * torch.eye(d)
    w = torch.linalg.solve(A, X_c.T @ y_c)
    return X_q @ w


def nw_predict(X_c, y_c, X_q, sigma=1.0):
    sq = ((X_q[:, None] - X_c[None, :]) ** 2).sum(-1)
    W = torch.exp(-sq / (sigma ** 2))
    W = W / W.sum(dim=1, keepdim=True).clamp_min(1e-9)
    return W @ y_c


lam = (prior.sigma_eps ** 2) / (prior.sigma_w ** 2)
mse = {'PFN': [], 'Bayes': [], 'Ridge': [], 'NW': []}
with torch.no_grad():
    for k in range(50):
        X_c, y_c, X_q, y_q = prior.sample_episode(n_ctx=64, n_query=32, d=D, seed=20000 + k)
        yhat_pfn = model(X_q, X_c, y_c).squeeze(-1)
        yhat_bay = bayes_posterior_mean(X_c, y_c, X_q,
                                        sigma_w=prior.sigma_w, sigma_eps=prior.sigma_eps)
        yhat_rid = ridge_predict(X_c, y_c, X_q, lam=lam)
        yhat_nw  = nw_predict(X_c, y_c, X_q, sigma=1.0)
        for name, yhat in [('PFN', yhat_pfn), ('Bayes', yhat_bay),
                           ('Ridge', yhat_rid), ('NW', yhat_nw)]:
            mse[name].append(((yhat - y_q) ** 2).mean().item())

means = {k: float(np.mean(v)) for k, v in mse.items()}
stds  = {k: float(np.std(v) / np.sqrt(len(v))) for k, v in mse.items()}
print('held-out MSE (mean over 50 tasks, +/- SEM):')
for k in ['PFN', 'Bayes', 'Ridge', 'NW']:
    print(f'  {k:<6} {means[k]:.4f} +/- {stds[k]:.4f}')

fig, ax = plt.subplots(1, 1, figsize=(6.5, 3.5))
keys = ['PFN', 'Bayes', 'Ridge', 'NW']
xs = np.arange(len(keys))
ax.bar(xs, [means[k] for k in keys], yerr=[stds[k] for k in keys],
       color=['C0', 'C2', 'C3', 'C4'], capsize=4, alpha=0.85)
ax.set_xticks(xs); ax.set_xticklabels(keys)
ax.set_ylabel('held-out MSE')
ax.set_title('Figure 19.3: Held-out MSE by predictor (lower is better)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_19_03_pfn_holdout_mse.pdf', bbox_inches='tight')
plt.show()
